### import Dependencies

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, MatchAny, FieldCondition, Filter, Prefetch, FusionQuery

import pandas as pd
import numpy as np
import openai
import json
import tiktoken

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Retrieve all item IDs from Amazon Items Qdrant Collection

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [3]:
dummy_vector = np.zeros(1536).tolist()

In [5]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
)

In [6]:
payload.points

[ScoredPoint(id=492, version=2, score=0.0, payload={'parent_asin': 'B0BFGXRMJN'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=385, version=2, score=0.0, payload={'parent_asin': 'B09SYSRBLC'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=386, version=2, score=0.0, payload={'parent_asin': 'B09Z2NG36C'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=389, version=2, score=0.0, payload={'parent_asin': 'B09XCJZKSR'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=330, version=2, score=0.0, payload={'parent_asin': 'B09ZPMPNVH'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=148, version=2, score=0.0, payload={'parent_asin': 'B0CGHVQ4M3'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=293, version=2, score=0.0, payload={'parent_asin': 'B0C3X7JK7R'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=375, version=2, score=0.0, payload={'parent_asin': 'B09VDNKL4G'}, vector=N

In [7]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]

In [8]:
parent_asin_list

['B0BFGXRMJN',
 'B09SYSRBLC',
 'B09Z2NG36C',
 'B09XCJZKSR',
 'B09ZPMPNVH',
 'B0CGHVQ4M3',
 'B0C3X7JK7R',
 'B09VDNKL4G',
 'B0BGJ2TDXV',
 'B0BHSPB4JL',
 'B0BP68QCZ2',
 'B09TR8VXGL',
 'B0CGM29ZL7',
 'B0C9XFF3CT',
 'B0BR33XH8D',
 'B0B45VPMT4',
 'B0B3GLJSFY',
 'B09Y39SFFL',
 'B0BHWK4T9J',
 'B09WR4KR4X',
 'B0C7492QYK',
 'B0BZQ5YKKY',
 'B09T7B55W3',
 'B09WDJ1NNB',
 'B0BPS9F24C',
 'B0B1TP23DX',
 'B09W9DV8PX',
 'B0BLJXPRL9',
 'B0BBG6P5SM',
 'B0BG55JZ1T',
 'B0BN8Q1Z9S',
 'B0BJZV7QVF',
 'B0BTSXW6NX',
 'B09T6N7TPS',
 'B0C3XYD574',
 'B08VZ2X1H9',
 'B0B1ZJPY3K',
 'B0CDWZPRF2',
 'B0B8ZHF99W',
 'B0CBMPG524',
 'B0B96LV4C5',
 'B0BJJYFRLF',
 'B0B7MCLH1Y',
 'B0BFDH4NX1',
 'B09V7MDRZ1',
 'B0B68DHY1J',
 'B0B5TJJTPY',
 'B09RKH6ST3',
 'B09KZK1494',
 'B09QPH2GN3',
 'B09YRG3BYM',
 'B09XDNPQHZ',
 'B0BKG66J4N',
 'B09FFFSH25',
 'B09YL52R4H',
 'B09Q5W9HPQ',
 'B0BJ1MG9JL',
 'B09YRMGZDS',
 'B099K711GD',
 'B0BNHVLF7G',
 'B0C5CLD1HF',
 'B0C37GVWBP',
 'B0BQYN54PZ',
 'B0BML99MQ3',
 'B0C7HFJR2K',
 'B09SFN9NRX',
 'B09Q345B

In [9]:
len(parent_asin_list)

500

### Load Amazon Reviews Dataset

In [12]:
df_reviews = pd.read_json("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [13]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1,love it but ... spoke too soon,UPDATE: OMGGG this thing SKIPSSSS and crackle...,[],B07QK824QM,B0CBWV3Q8B,AHGAOIZVODNHYMNCBV4DECZH42UQ,2019-12-05 17:36:37.115,4,True
1,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True
2,3,Not For Larger Wrists,It's been a while since I had gotten a new ban...,[{'small_image_url': 'https://m.media-amazon.c...,B09GVKXFRQ,B0B19SL9TF,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2021-11-18 17:02:02.035,1,False
3,5,Beautifully Made,This is another beautifully made bag from Vera...,[],B08TPSL1FW,B0B85V929L,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2021-07-07 22:44:26.631,0,False
4,5,So easy to use,Overall I am enjoying my new Skylight frame. ...,[],B01N7ENHO6,B0BV3FZ8KY,AFIDB4NPTG3E7KHCX7BUO4WDE6LQ,2019-12-31 20:59:55.824,0,True


In [14]:
len(df_reviews)

96850

In [15]:
df_reviews_sample = df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [16]:
len(df_reviews_sample)

3943

### Define functions to preprocess reviews data

In [17]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [18]:
encoding = tiktoken.encoding_for_model("text-embedding-3-small")

In [19]:
encoding.encode("Can I gert some earphones?")

[6854, 358, 342, 531, 1063, 2487, 17144, 30]

In [20]:
def token_count(row, model="text-embedding-3-small"):

    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(row["preprocessed_data"]))

In [21]:
df_reviews_sample["preprocessed_data"] = df_reviews_sample.apply(preprocess_reviews_data, axis=1)

/var/folders/hd/mwwk_4zs7blcj6tc0ycm9t5c0000gn/T/ipykernel_2589/1882656165.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reviews_sample["preprocessed_data"] = df_reviews_sample.apply(preprocess_reviews_data, axis=1)


In [22]:
df_reviews_sample["preprocessed_data_token_count"] = df_reviews_sample.apply(token_count, axis=1)

/var/folders/hd/mwwk_4zs7blcj6tc0ycm9t5c0000gn/T/ipykernel_2589/1530805122.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reviews_sample["preprocessed_data_token_count"] = df_reviews_sample.apply(token_count, axis=1)


In [23]:
df_reviews_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,preprocessed_data_token_count
17,5,Five Stars,Worked perfect and was no problem to set up th...,[],B003CFATNI,B0BJ1SLJH1,AE5PZITMWZJBPVKL7XXYUFBTITQA,2017-02-11 03:35:13.000,0,True,Five Stars Worked perfect and was no problem t...,20
29,5,Works Fine,"This card is very small, I was surprised when ...",[],B003CFATNI,B0BJ1SLJH1,AF6NMYYHILKNULG772IZLOFGYAAA,2013-02-24 05:02:21.000,2,True,"Works Fine This card is very small, I was surp...",44
105,5,Huge,Love it. Big and wide. Great clarity. I use it...,[],B09ZYTFX33,B0BMQP624X,AFOHEEK7Z7O37JDYNEMCZSBUZHYA,2022-09-22 20:52:44.420,1,True,Huge Love it. Big and wide. Great clarity. I u...,30
107,5,Does what it says its does,Works like a champ,[],B003CFATNI,B0BJ1SLJH1,AGUWDJXPNE4CB2AFX4YK76XZUHGQ,2019-10-09 04:45:31.060,0,True,Does what it says its does Works like a champ,10
120,5,Awesome,Works awesome! I have no complaints installed ...,[],B003CFATNI,B0BJ1SLJH1,AF3IOPFWELTFPZFYBKFWOJ3B6D7Q,2014-03-06 06:06:16.000,0,True,Awesome Works awesome! I have no complaints in...,27


In [24]:
len(df_reviews_sample)

3943

In [25]:
df_reviews_sample = df_reviews_sample[df_reviews_sample["preprocessed_data_token_count"] < 8192]

In [26]:
len(df_reviews_sample)

3943

In [27]:
total_tokens = df_reviews_sample["preprocessed_data_token_count"].sum()

In [28]:
total_tokens

np.int64(231141)

### Create a new Qdrant collection for reviews

In [29]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-reviews",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

True

In [30]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-reviews",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding functions

In [31]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

In [32]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Embed the text and add additional fields to the payload of each vector for reviews

In [33]:
data_to_embed_reviews = df_reviews_sample[["preprocessed_data", "parent_asin"]].to_dict(orient="records")

In [34]:
data_to_embed_reviews

[{'preprocessed_data': 'Five Stars Worked perfect and was no problem to set up the drivers. Well worth the money.',
  'parent_asin': 'B0BJ1SLJH1'},
 {'preprocessed_data': 'Works Fine This card is very small, I was surprised when I saw it.  It uses a Realtek chip so windows had no problem recognizing it and I expect it to last for as long as I need it.',
  'parent_asin': 'B0BJ1SLJH1'},
 {'preprocessed_data': 'Huge Love it. Big and wide. Great clarity. I use it for work - email, spreadsheets, applications. Extra space improves productivity.',
  'parent_asin': 'B0BMQP624X'},
 {'preprocessed_data': 'Does what it says its does Works like a champ',
  'parent_asin': 'B0BJ1SLJH1'},
 {'preprocessed_data': 'Awesome Works awesome! I have no complaints installed the card on my Win 7. It works incredibly fast and moves files fast LOVE it',
  'parent_asin': 'B0BJ1SLJH1'},
 {'preprocessed_data': 'No Mac Address This card did not come with a MAC address and would not take a static IP or get one from D

In [35]:
text_to_embed_reviews = [data["preprocessed_data"] for data in data_to_embed_reviews]

In [36]:
text_to_embed_reviews

['Five Stars Worked perfect and was no problem to set up the drivers. Well worth the money.',
 'Works Fine This card is very small, I was surprised when I saw it.  It uses a Realtek chip so windows had no problem recognizing it and I expect it to last for as long as I need it.',
 'Huge Love it. Big and wide. Great clarity. I use it for work - email, spreadsheets, applications. Extra space improves productivity.',
 'Does what it says its does Works like a champ',
 'Awesome Works awesome! I have no complaints installed the card on my Win 7. It works incredibly fast and moves files fast LOVE it',
 'No Mac Address This card did not come with a MAC address and would not take a static IP or get one from DHCP.  Returned.',
 'Great noise cancellation and bass, clear sound, ok treble These headphones offer crisp sound with deep bass capability, and decent treble that can *sometimes sound slightly tinny but isn’t a big deal. The noise cancellation works great on a wide range of white noise, from

In [37]:
embeddings_reviews = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

Processed 500 of 3943
Processed 1000 of 3943
Processed 1500 of 3943
Processed 2000 of 3943
Processed 2500 of 3943
Processed 3000 of 3943
Processed 3500 of 3943
Processed 4000 of 3943


In [38]:
len(embeddings_reviews)

3943

In [39]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings_reviews, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={
                "text": data["preprocessed_data"],
                "parent_asin": data["parent_asin"],
            }
        )
    )
    i += 1

In [40]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-items-collection-01-reviews",
        wait=True,
        points=batch
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

Processed 100 of 3943
Processed 200 of 3943
Processed 300 of 3943
Processed 400 of 3943
Processed 500 of 3943
Processed 600 of 3943
Processed 700 of 3943
Processed 800 of 3943
Processed 900 of 3943
Processed 1000 of 3943
Processed 1100 of 3943
Processed 1200 of 3943
Processed 1300 of 3943
Processed 1400 of 3943
Processed 1500 of 3943
Processed 1600 of 3943
Processed 1700 of 3943
Processed 1800 of 3943
Processed 1900 of 3943
Processed 2000 of 3943
Processed 2100 of 3943
Processed 2200 of 3943
Processed 2300 of 3943
Processed 2400 of 3943
Processed 2500 of 3943
Processed 2600 of 3943
Processed 2700 of 3943
Processed 2800 of 3943
Processed 2900 of 3943
Processed 3000 of 3943
Processed 3100 of 3943
Processed 3200 of 3943
Processed 3300 of 3943
Processed 3400 of 3943
Processed 3500 of 3943
Processed 3600 of 3943
Processed 3700 of 3943
Processed 3800 of 3943
Processed 3900 of 3943
Processed 4000 of 3943


### A function to run search against reviews on a prefiltered set of product IDs

In [41]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-reviews",
        prefetch=[
            Prefetch(
                query=query_embedding,
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    return results

In [45]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B0BWVD7KZG"])

In [46]:
reviews.points

[ScoredPoint(id=3229, version=34, score=0.5, payload={'text': 'Band has broken twice Poor Band quality', 'parent_asin': 'B0BWVD7KZG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=189, version=3, score=0.33333334, payload={'text': "Junky piece What a cheap piece. I wouldn't waste your money.", 'parent_asin': 'B0BWVD7KZG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=664, version=8, score=0.25, payload={'text': 'Cheap quality I’ve only had this watch for one month and it broke already! The hook came off completely. Very poorly made.', 'parent_asin': 'B0BWVD7KZG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3369, version=35, score=0.2, payload={'text': 'Poor quality watches I bought two watches for my kids one month ago, the battery is shortly use, only 1-2days needs to charge, finally its were not working just one month it totally broke without any attacking, but it already pass the deadline.', 'parent_asin': 'B0BWVD7KZG'}, vec

In [47]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B0BWVD7KZG", "B0BGJ2TDXV"])

In [48]:
reviews.points

[ScoredPoint(id=3229, version=34, score=0.5, payload={'text': 'Band has broken twice Poor Band quality', 'parent_asin': 'B0BWVD7KZG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=189, version=3, score=0.33333334, payload={'text': "Junky piece What a cheap piece. I wouldn't waste your money.", 'parent_asin': 'B0BWVD7KZG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=664, version=8, score=0.25, payload={'text': 'Cheap quality I’ve only had this watch for one month and it broke already! The hook came off completely. Very poorly made.', 'parent_asin': 'B0BWVD7KZG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3369, version=35, score=0.2, payload={'text': 'Poor quality watches I bought two watches for my kids one month ago, the battery is shortly use, only 1-2days needs to charge, finally its were not working just one month it totally broke without any attacking, but it already pass the deadline.', 'parent_asin': 'B0BWVD7KZG'}, vec

In [52]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B0BWVD7KZG", "B0BGJ2TDXV"], k=20)

In [53]:
for point in reviews.points:
    print(point.payload["parent_asin"])
    print(point.payload["text"])
    print("-"*100)

B0BWVD7KZG
Band has broken twice Poor Band quality
----------------------------------------------------------------------------------------------------
B0BWVD7KZG
Junky piece What a cheap piece. I wouldn't waste your money.
----------------------------------------------------------------------------------------------------
B0BWVD7KZG
Cheap quality I’ve only had this watch for one month and it broke already! The hook came off completely. Very poorly made.
----------------------------------------------------------------------------------------------------
B0BWVD7KZG
Poor quality watches I bought two watches for my kids one month ago, the battery is shortly use, only 1-2days needs to charge, finally its were not working just one month it totally broke without any attacking, but it already pass the deadline.
----------------------------------------------------------------------------------------------------
B0BWVD7KZG
It's junk very unhappy Junk
--------------------------------------------